# Nigeria DHS 2024 climate-linkage pilot
## December 2023 interviews, September–November 2023 climate

Open this notebook in Colab and choose **Runtime → Run all**. At each upload prompt, choose one file from any folder on your computer:

1. `Nigeria_DHS_WASH_Households.csv.gz` from the harmonisation notebook
2. `chirps-v3.0.2023.09.tif`
3. `chirps-v3.0.2023.10.tif`
4. `chirps-v3.0.2023.11.tif`
5. `ERA5_Data.zip`, with the original NetCDF member files inside

The expected pilot has **173 cluster-month records** and **4,902 household records**. Upload only data you are authorised to process in your Colab session. The household file contains DHS identifiers and displaced coordinates. Download your results before resetting the runtime.

## 1. Choose the files

Each prompt saves the selected file under its expected name in this session. The following cells check file format, the household records and the temperature timestamps. If you stop partway, run this cell again to continue from the files already uploaded. To replace a wrong file, delete that file under `/content/climate_pilot_inputs` in Colab’s **Files** panel and rerun from this cell.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import re
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
from PIL import Image

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

MONTHS = pd.period_range("2023-09", "2023-11", freq="M")
INTERVIEW_MONTH = pd.Timestamp("2023-12-01")
RAIN_NAMES = [f"chirps-v3.0.{month.year}.{month.month:02d}.tif" for month in MONTHS]
INPUT_NAMES = ["Nigeria_DHS_WASH_Households.csv.gz", *RAIN_NAMES, "ERA5_Data.zip"]
KEY = ["survey_round", "hv001", "interview_month"]

INPUT_DIR = Path("/content/climate_pilot_inputs" if files else "climate_pilot_inputs")
OUTPUT_DIR = Path("/content/climate_pilot_outputs" if files else "climate_pilot_outputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

for number, name in enumerate(INPUT_NAMES, start=1):
    path = INPUT_DIR / name
    if not path.is_file() and files is not None:
        print(f"[{number}/5] Choose {name}", flush=True)
        partial = path.with_suffix(path.suffix + ".part")
        try:
            files.upload_file(str(partial))
            partial.replace(path)
        finally:
            partial.unlink(missing_ok=True)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty input: {path}")
    print(f"[{number}/5] Ready: {name}")

## 2. Select December interviews and make one row per cluster

`survey_round` identifies the DHS release. `interview_month` identifies when each interview took place. The December 2023 interviews are part of the 2024 release. Several households share a cluster, so we only need to extract the climate data once per cluster for this month.

In [ ]:
households = pd.read_csv(INPUT_DIR / INPUT_NAMES[0], parse_dates=["interview_month"])
required = set(KEY + ["hv002", "gps_usable", "cluster_latitude", "cluster_longitude", "residence"])
if missing := required - set(households.columns):
    raise ValueError(f"Household export is missing columns: {sorted(missing)}")

pilot = households.loc[
    households.survey_round.eq(2024) & households.interview_month.eq(INTERVIEW_MONTH)
].copy()
assert len(pilot) == 4902, f"Expected 4,902 December households, found {len(pilot):,}."
assert pilot[KEY + ["hv002"]].notna().all().all()
assert not pilot.duplicated(KEY + ["hv002"]).any()

eligible = pilot.gps_usable.astype(str).str.strip().str.lower().eq("true")
site_columns = ["cluster_latitude", "cluster_longitude", "residence"]
sites = pilot.loc[eligible, KEY + site_columns].drop_duplicates().reset_index(drop=True)
assert not sites.duplicated(KEY).any(), "A cluster-month has inconsistent location or residence."
assert len(sites) == 173 and eligible.all(), "Check the expected December GPS coverage."
assert sites.residence.isin(["Urban", "Rural"]).all()
assert sites.cluster_latitude.between(4, 14).all()
assert sites.cluster_longitude.between(2, 15).all()
sites["buffer_km"] = sites.residence.map({"Urban": 2, "Rural": 5})
print(f"{len(pilot):,} households and {len(sites):,} unique cluster-months.")

## 3. Read the rainfall and temperature grids

CHIRPS is a monthly rainfall total in millimetres. ERA5-Land `t2m` is monthly mean air temperature at 2 metres. The code checks that the temperature file is in Kelvin or Celsius, reads the actual time coordinate, and selects the three required months. It accepts one NetCDF containing all months or separate NetCDFs in the ZIP. If you downloaded GRIB, use the CDS form to request **NetCDF** and select **Monthly averaged reanalysis**. Extra dates in the ZIP are skipped. Duplicate required months are flagged.

A GeoTIFF or NetCDF stores data as a grid. These readers check the regular grid structure before using it. The CHIRPS names are used as month labels, so select the correct file at each prompt.

In [ ]:
from dataclasses import dataclass

@dataclass
class Grid:
    values: np.ndarray  # Rows run north to south. Columns run west to east.
    west: float
    north: float
    dx: float
    dy: float

def read_rain(path):
    with Image.open(path) as image:
        if image.format != "TIFF" or image.n_frames != 1:
            raise ValueError(f"Expected a single-band GeoTIFF: {path.name}")
        tags = image.tag_v2
        scale, tie = tags.get(33550), tags.get(33922)
        if scale is None or tie is None or 34264 in tags:
            raise ValueError(f"Expected a north-up GeoTIFF with a pixel scale and tie point: {path.name}")
        dx, dy = map(float, scale[:2])
        col, row, _, lon, lat, _ = map(float, tie[:6])
        keys = tags.get(34735, ())
        geokeys = {
            key: value for key, location, _, value in
            (keys[i:i + 4] for i in range(4, len(keys), 4)) if location == 0
        }
        if geokeys.get(2048, 4326) != 4326 or geokeys.get(1024, 2) != 2:
            raise ValueError(f"Expected a WGS84 longitude/latitude grid: {path.name}")
        west, north = lon - col * dx, lat + row * dy
        if geokeys.get(1025, 1) == 2:  # A point tie refers to a cell centre.
            west -= dx / 2
            north += dy / 2
        values = np.array(image, dtype=np.float64)
        nodata = tags.get(42113)
        if nodata is not None:
            values[values == float(nodata)] = np.nan
    if values.ndim != 2 or dx <= 0 or dy <= 0:
        raise ValueError(f"Unexpected CHIRPS grid: {path.name}")
    values[values < 0] = np.nan  # Negative rainfall indicates missing data.
    return Grid(values, west, north, dx, dy)

def attribute(variable, name, default=None):
    value = getattr(variable, name, default)
    return value.decode() if isinstance(value, bytes) else value

def read_numeric(variable):
    raw = np.ma.asarray(variable[:], dtype=np.float64).filled(np.nan)
    for name in ("_FillValue", "missing_value"):
        for invalid in np.atleast_1d(attribute(variable, name, np.nan)):
            raw[raw == invalid] = np.nan
    return raw * attribute(variable, "scale_factor", 1) + attribute(variable, "add_offset", 0)

def open_netcdf(path):
    with path.open("rb") as handle:
        signature = handle.read(8)
    if signature[:3] == b"CDF" and signature[3] in (1, 2):
        from scipy.io import netcdf_file
        return netcdf_file(path, mmap=False)
    if signature == b"\x89HDF\r\n\x1a\n" or signature[:3] == b"CDF":
        try:
            import netCDF4
        except ModuleNotFoundError:
            if files is None:
                raise RuntimeError("Install netCDF4 to read this NetCDF format.")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "netCDF4"])
            import netCDF4
        dataset = netCDF4.Dataset(path)
        dataset.set_auto_maskandscale(False)  # read_numeric applies it once.
        return dataset
    raise ValueError(f"{path.name} is not a supported NetCDF file. Request NetCDF in CDS.")

def read_months(variable):
    units = attribute(variable, "units", "")
    match = re.fullmatch(r"(seconds?|minutes?|hours?|days?) since (.+)", units, flags=re.I)
    calendar = attribute(variable, "calendar", "standard")
    if match is None or calendar not in ("standard", "gregorian", "proleptic_gregorian"):
        raise ValueError(f"Unsupported ERA5 time metadata: {units!r}, {calendar!r}")
    unit = {"second": "s", "minute": "min", "hour": "h", "day": "D"}[match[1].lower().rstrip("s")]
    dates = pd.to_datetime(
        pd.Timestamp(match[2]) + pd.to_timedelta(read_numeric(variable), unit=unit),
        utc=True,
    ).tz_convert(None)
    return dates.to_period("M")

def temperature_grids(path):
    with open_netcdf(path) as nc:
        v = nc.variables
        name = next((key for key in ("t2m", "2m_temperature", "tas") if key in v), None)
        if name is None:
            raise ValueError(f"No 2m air temperature variable in {path.name}.")
        temp = v[name]
        dims = list(temp.dimensions)
        time = next((key for key in ("valid_time", "time") if key in v and len(v[key].dimensions) == 1), None)
        lat = next((key for key in ("latitude", "lat") if key in v and len(v[key].dimensions) == 1), None)
        lon = next((key for key in ("longitude", "lon") if key in v and len(v[key].dimensions) == 1), None)
        if None in (time, lat, lon):
            raise ValueError("Expected one-dimensional time, latitude and longitude coordinates.")
        wanted = [v[key].dimensions[0] for key in (time, lat, lon)]
        if len(set(wanted)) != 3 or not set(wanted) <= set(dims):
            raise ValueError(f"Unexpected temperature dimensions: {dims}")
        values = read_numeric(temp)
        for axis in range(len(dims) - 1, -1, -1):
            if dims[axis] not in wanted:
                if values.shape[axis] != 1:
                    raise ValueError(f"Choose monthly averaged reanalysis. Extra dimension: {dims[axis]}")
                values = values.squeeze(axis=axis)
                dims.pop(axis)
        values = values.transpose([dims.index(key) for key in wanted])
        units = attribute(temp, "units", "").strip().lower()
        if units in ("k", "kelvin"):
            values -= 273.15
        elif units not in ("c", "degc", "degree_celsius", "degrees_celsius", "°c", "celsius"):
            raise ValueError(f"Review 2m temperature units: {units!r}")

        lat_values = read_numeric(v[lat])
        lon_values = (read_numeric(v[lon]) + 180) % 360 - 180
        iy, ix = np.argsort(lat_values)[::-1], np.argsort(lon_values)
        y, x = lat_values[iy], lon_values[ix]
        if len(x) < 2 or len(y) < 2:
            raise ValueError("Temperature grid needs at least two rows and columns.")
        dx, dy = x[1] - x[0], y[0] - y[1]
        if dx <= 0 or dy <= 0 or not np.allclose(np.diff(x), dx, atol=1e-5) or not np.allclose(-np.diff(y), dy, atol=1e-5):
            raise ValueError("Expected a regular ERA5-Land longitude/latitude grid.")
        dates = read_months(v[time])
        return [
            (month, Grid(values[i][np.ix_(iy, ix)], x[0] - dx / 2, y[0] + dy / 2, dx, dy))
            for i, month in enumerate(dates) if month in MONTHS
        ]

rain_grids = {month: read_rain(INPUT_DIR / name) for month, name in zip(MONTHS, RAIN_NAMES)}
temp_grids = {}
era_dir = INPUT_DIR / "era_netcdf"
era_dir.mkdir(exist_ok=True)
with ZipFile(INPUT_DIR / "ERA5_Data.zip") as archive:
    members = [m for m in archive.namelist() if Path(m).suffix.lower() in (".nc", ".nc4", ".netcdf")]
    if not members:
        raise ValueError("ERA5_Data.zip has no NetCDF members. Request NetCDF in the CDS download form.")
    for i, member in enumerate(members):
        target = era_dir / f"member_{i}.nc"
        with archive.open(member) as source, target.open("wb") as destination:
            shutil.copyfileobj(source, destination)
        for month, grid in temperature_grids(target):
            if month in temp_grids:
                raise ValueError(f"More than one temperature layer for {month}. Select one monthly product.")
            temp_grids[month] = grid

if missing_months := set(MONTHS) - set(temp_grids):
    raise ValueError(f"Missing ERA5-Land months: {sorted(map(str, missing_months))}")
print("Rainfall and temperature grids ready:", ", ".join(map(str, MONTHS)))

## 4. Average the climate values inside each buffer

A buffer is a circle around the displaced cluster position. The code places sample points on a regular **200 m grid** inside the circle. Each point takes the climate value from the grid cell it falls in. Averaging those values approximates an area-weighted buffer mean, even when the climate grid is wider than the buffer. The method does not increase the underlying spatial resolution.

The pilot uses **2 km urban** and **5 km rural** buffers. A monthly estimate is accepted only if at least **50%** of sample points contain valid grid values. This is a pilot quality rule, not a DHS standard. Retain the coverage fractions to review coastal and missing-data cases. There is no automatic fallback to a point value.

In [ ]:
SAMPLE_SPACING_M = 200
MIN_VALID_FRACTION = 0.50

def disc_offsets(radius_m):
    axis = np.arange(-radius_m, radius_m + SAMPLE_SPACING_M, SAMPLE_SPACING_M)
    east, north = np.meshgrid(axis, axis)
    inside = east**2 + north**2 <= radius_m**2
    return east[inside], north[inside]

OFFSETS = {km: disc_offsets(km * 1000) for km in (2, 5)}

def sample_locations(latitude, longitude, radius_km):
    east, north = OFFSETS[radius_km]
    bearing = np.arctan2(east, north)
    angular_distance = np.hypot(east, north) / 6_371_008.8
    y, x = np.deg2rad([latitude, longitude])
    lat = np.arcsin(
        np.sin(y) * np.cos(angular_distance) +
        np.cos(y) * np.sin(angular_distance) * np.cos(bearing)
    )
    lon = x + np.arctan2(
        np.sin(bearing) * np.sin(angular_distance) * np.cos(y),
        np.cos(angular_distance) - np.sin(y) * np.sin(lat),
    )
    return np.rad2deg(lat), np.rad2deg(lon)

def buffer_mean(grid, latitudes, longitudes):
    rows = np.floor((grid.north - latitudes) / grid.dy).astype(int)
    cols = np.floor((longitudes - grid.west) / grid.dx).astype(int)
    nrows, ncols = grid.values.shape
    inside = (rows >= 0) & (rows < nrows) & (cols >= 0) & (cols < ncols)
    values = grid.values[rows[inside], cols[inside]]
    valid = values[np.isfinite(values)]
    fraction = valid.size / latitudes.size
    return (float(valid.mean()) if fraction >= MIN_VALID_FRACTION else np.nan), fraction

monthly_rows = []
for site in sites.itertuples(index=False):
    latitudes, longitudes = sample_locations(site.cluster_latitude, site.cluster_longitude, site.buffer_km)
    for month in MONTHS:
        rain, rain_fraction = buffer_mean(rain_grids[month], latitudes, longitudes)
        temp, temp_fraction = buffer_mean(temp_grids[month], latitudes, longitudes)
        monthly_rows.append({
            "survey_round": site.survey_round, "hv001": site.hv001,
            "interview_month": site.interview_month, "buffer_km": site.buffer_km,
            "climate_month": str(month), "days_in_month": month.days_in_month,
            "rain_mm": rain, "temp_c": temp,
            "rain_valid_fraction": rain_fraction, "temp_valid_fraction": temp_fraction,
        })
monthly = pd.DataFrame(monthly_rows)
assert len(monthly) == 519 and not monthly.duplicated(KEY + ["climate_month"]).any()
assert monthly.groupby(KEY).size().eq(3).all()
assert monthly.rain_mm.dropna().ge(0).all()
assert monthly.temp_c.dropna().between(-10, 60).all(), "Review ERA5-Land units or data range."
print("Extracted", len(monthly), "monthly cluster values for each variable.")

## 5. Make the three-month exposures and join them to households

Sum monthly rainfall. Weight temperature by the number of days in each month. A three-month exposure is left blank if any of its three monthly values is missing or falls below the coverage threshold.

The household merge is **many-to-one**. Many households can share a cluster-month, but the exposure table has only one row for that key.

In [ ]:
monthly["temp_degree_days"] = monthly.temp_c * monthly.days_in_month
grouped = monthly.groupby(KEY, sort=True)
exposures = grouped.agg(
    buffer_km=("buffer_km", "first"),
    exposure_start=("climate_month", "min"),
    exposure_end=("climate_month", "max"),
    rain_months_valid=("rain_mm", "count"),
    temp_months_valid=("temp_c", "count"),
    min_rain_valid_fraction=("rain_valid_fraction", "min"),
    min_temp_valid_fraction=("temp_valid_fraction", "min"),
)
exposures["rain_mm_3m"] = grouped.rain_mm.sum(min_count=3)
exposures["temp_mean_c_3m"] = grouped.temp_degree_days.sum(min_count=3) / grouped.days_in_month.sum()
exposures = exposures.reset_index()
assert len(exposures) == 173 and not exposures.duplicated(KEY).any()
assert exposures.exposure_start.eq("2023-09").all() and exposures.exposure_end.eq("2023-11").all()

joined = pilot.merge(exposures, on=KEY, how="left", validate="many_to_one", indicator=True)
assert len(joined) == len(pilot) and joined._merge.eq("both").all()
assert not joined.duplicated(KEY + ["hv002"]).any()
joined = joined.drop(columns="_merge")
print(f"Joined {len(exposures):,} cluster-month rows to {len(joined):,} households.")

## 6. Save and review the results

Open the Colab **Files** sidebar and download the four outputs in `/content/climate_pilot_outputs/`. Start with the QC file. It separates successful key matching from missing climate exposure. If either variable is incomplete, inspect the monthly file to find the affected clusters, months and coverage fractions.

- **Cluster-month CSV** for the three-month exposures and coverage flags.
- **Household CSV.GZ** for the December interviews with exposures attached.
- **Monthly audit CSV** for the three individual months per cluster.
- **QC CSV** for the counts to verify before scaling up.

The remaining 2024 interviews require climate data through **April 2024** and a separate three-month window for each interview month. This notebook deliberately validates the December subset first.

In [ ]:
qc = pd.DataFrame([{
    "survey_round": 2024, "interview_month": "2023-12",
    "exposure_start": "2023-09", "exposure_end": "2023-11",
    "households_before": len(pilot), "households_after": len(joined),
    "cluster_month_rows": len(exposures), "monthly_audit_rows": len(monthly),
    "clusters_missing_rain": int(exposures.rain_mm_3m.isna().sum()),
    "clusters_missing_temperature": int(exposures.temp_mean_c_3m.isna().sum()),
    "households_missing_rain": int(joined.rain_mm_3m.isna().sum()),
    "households_missing_temperature": int(joined.temp_mean_c_3m.isna().sum()),
    "sampling_spacing_m": SAMPLE_SPACING_M, "min_valid_fraction": MIN_VALID_FRACTION,
}])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exposures.to_csv(OUTPUT_DIR / "December_2023_Cluster_Climate.csv", index=False)
joined.to_csv(OUTPUT_DIR / "December_2023_Households_Climate.csv.gz", index=False, compression="gzip")
monthly.to_csv(OUTPUT_DIR / "December_2023_Monthly_Climate_Audit.csv", index=False)
qc.to_csv(OUTPUT_DIR / "December_2023_Climate_QC.csv", index=False)
print(qc.T.to_string(header=False))
print(f"Saved four files in {OUTPUT_DIR.resolve()}")

### Data sources

- [CHIRPS v3 Africa monthly GeoTIFFs](https://data.chc.ucsb.edu/products/CHIRPS/v3.0/monthly/africa/tifs/)
- [Copernicus ERA5-Land monthly averaged data](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-monthly-means)
- [DHS guidance on geospatial data and displacement](https://dhsprogram.com/Methodology/GPS-Data.cfm)